# Retrieve Statcast Pitch-by-Pitch Data (2021–2025)

Pulls one season at a time from the pybaseball Statcast endpoint and saves
each year to a separate CSV in this directory.  The file-exists check makes
the notebook safe to re-run — already-saved seasons are skipped.

In [ ]:
import os
import time
import pandas as pd
import pybaseball

pybaseball.cache.enable()

OUT_DIR = os.path.dirname(os.path.abspath("__file__"))   # same folder as this notebook
print(f"Output directory: {OUT_DIR}")

In [ ]:
# MLB regular-season date ranges (broad end dates capture all regular-season games)
SEASONS = {
    2021: ("2021-04-01", "2021-10-04"),   # full 162-game season
    2022: ("2022-04-07", "2022-10-06"),
    2023: ("2023-03-30", "2023-10-02"),
    2024: ("2024-03-20", "2024-09-30"),   # Seoul series opened March 20
    2025: ("2025-03-27", "2025-09-29"),
}

In [ ]:
def fetch_season(year: int, start: str, end: str, out_dir: str) -> str:
    """
    Pull all Statcast pitches for a season and save to CSV.
    Skips if the file already exists.
    Returns the output file path.
    """
    out_path = os.path.join(out_dir, f"full_pitching_data_{year}.csv")

    if os.path.exists(out_path):
        size_mb = os.path.getsize(out_path) / 1024 / 1024
        print(f"{year}: already saved → {out_path} ({size_mb:.1f} MB)  [skipped]")
        return out_path

    print(f"{year}: fetching {start} → {end} ...")
    t0 = time.time()

    # statcast() chunks large ranges automatically; verbose=False suppresses per-chunk prints
    df = pybaseball.statcast(start_dt=start, end_dt=end, verbose=True)

    elapsed = time.time() - t0
    print(f"{year}: fetched {len(df):,} pitches in {elapsed:.0f}s")

    df.to_csv(out_path, index=False)
    size_mb = os.path.getsize(out_path) / 1024 / 1024
    print(f"{year}: saved → {out_path} ({size_mb:.1f} MB)")
    return out_path

In [ ]:
# Pull each season — re-run any time; completed seasons are skipped automatically
for year, (start, end) in sorted(SEASONS.items()):
    fetch_season(year, start, end, OUT_DIR)
    print()

In [ ]:
# Quick sanity check — print row counts for all saved files
print("Saved files:")
for year in sorted(SEASONS):
    path = os.path.join(OUT_DIR, f"full_pitching_data_{year}.csv")
    if os.path.exists(path):
        df = pd.read_csv(path, nrows=0)   # just read headers for column count
        row_count = sum(1 for _ in open(path)) - 1   # fast line count
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  {year}: {row_count:>8,} rows  {size_mb:6.1f} MB  cols={len(df.columns)}")
    else:
        print(f"  {year}: not found")